In [22]:
from pathlib import Path
import ipaddress

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, precision_recall_curve, auc
from lightgbm import LGBMClassifier

In [23]:
csv_path = Path("/content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/all_pcaps_device_rows_merged.csv")
df = pd.read_csv(csv_path, low_memory=False)

print("raw shape:", df.shape)
print(df.columns.tolist())
display(df.head())


raw shape: (214040, 19)
['frame_time_epoch', 'src_ip', 'dst_ip', 'ip_proto', 'tcp_srcport', 'tcp_dstport', 'udp_srcport', 'udp_dstport', 'frame_len', 'tcp_flags', 'ip_dsfield', 'tcp_stream', 'udp_stream', 'mqtt_topic', 'info', 'device_name', 'mapping_source', 'mapping_confidence', 'pcap_name']


,frame_time_epoch,src_ip,dst_ip,ip_proto,tcp_srcport,tcp_dstport,udp_srcport,udp_dstport,frame_len,tcp_flags,ip_dsfield,tcp_stream,udp_stream,mqtt_topic,info,device_name,mapping_source,mapping_confidence,pcap_name
0,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,NaN,NaN,1516,0x0010,0x02,1,NaN,/iot/locator,"Publish Message (id=12526) [/iot/locator], Pub...",IoT_Thermostat,topic,high,normal_10.pcap
1,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,NaN,NaN,1516,0x0010,0x02,1,NaN,/smarthome/thermostat,Publish Message (id=12535) [/smarthome/thermos...,IoT_Thermostat,topic,high,normal_10.pcap
2,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,NaN,NaN,523,0x0018,0x02,1,NaN,/smarthome/garageDoor,Publish Message (id=12546) [/smarthome/garageD...,IoT_Thermostat,topic,high,normal_10.pcap
3,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,NaN,NaN,223,0x0018,0x02,1,NaN,/smarthome/motionLights,Publish Message (id=12549) [/smarthome/motionL...,IoT_Motion_Light,topic,high,normal_10.pcap
4,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,NaN,NaN,350,0x0018,0x02,1,NaN,/smarfactory/Modbus-data,Publish Message (id=12550) [/smarfactory/Modbu...,IoT_Motion_Light,topic,high,normal_10.pcap


In [25]:
def pick_pcap_col(df):
    names = ["pcap_name", "source", "pcap", "file", "source_file", "pcap_file", "trace"]
    for name in names:
        if name in df.columns:
            return name

    df["_pcap_group"] = "single_source"
    return "_pcap_group"


def fix_device_name(x):
    text = str(x).strip()
    if text == "" or text.lower() == "nan":
        return "UNKNOWN_DEVICE"
    return text


def read_flag(x):
    text = str(x).strip()

    if text == "" or text.lower() == "nan" or text == "UNK":
        return 0

    try:
        if text.lower().startswith("0x"):
            return int(text, 16)
        return int(float(text))
    except Exception:
        return 0


def merge_flags(values):
    out = 0
    for v in values:
        out |= int(v)
    return out


def to_ipv4_net(x):
    text = str(x).strip()

    try:
        ip = ipaddress.ip_address(text)
    except Exception:
        return "UNK"

    if ip.version != 4:
        return text

    parts = text.split(".")
    return ".".join(parts[:3]) + ".0/24"


def split_in_time(df, ratio=0.7):
    data = df.sort_values("mid_time").reset_index(drop=True)

    cut = int(np.floor(len(data) * ratio))
    cut = max(cut, 1)

    if cut >= len(data):
        cut = len(data) - 1

    train_part = data.iloc[:cut].copy()
    test_part = data.iloc[cut:].copy()
    return train_part, test_part

In [26]:
work = df.copy()

pcap_col = pick_pcap_col(work)
print("pcap column:", pcap_col)

work["device_name"] = work["device_name"].apply(fix_device_name)

num_defaults = {
    "frame_time_epoch": 0.0,
    "frame_len": 0.0,
    "ip_proto": -1,
    "tcp_srcport": -1,
    "tcp_dstport": -1,
    "udp_srcport": -1,
    "udp_dstport": -1,
    "tcp_stream": -1,
    "udp_stream": -1,
}

for col, default in num_defaults.items():
    if col in work.columns:
        work[col] = pd.to_numeric(work[col], errors="coerce").fillna(default)
    else:
        work[col] = default

for col in ["src_ip", "dst_ip", "tcp_flags", "ip_dsfield", "mqtt_topic", pcap_col]:
    if col in work.columns:
        work[col] = work[col].fillna("UNK").astype(str)
    else:
        work[col] = "UNK"

work["src_port"] = work["tcp_srcport"].where(work["tcp_srcport"] != -1, work["udp_srcport"])
work["dst_port"] = work["tcp_dstport"].where(work["tcp_dstport"] != -1, work["udp_dstport"])

work["src_port"] = work["src_port"].fillna(-1).astype(int)
work["dst_port"] = work["dst_port"].fillna(-1).astype(int)
work["ip_proto"] = work["ip_proto"].fillna(-1).astype(int)

display(work.head())

pcap column: pcap_name


,frame_time_epoch,src_ip,dst_ip,ip_proto,tcp_srcport,tcp_dstport,udp_srcport,udp_dstport,frame_len,tcp_flags,...,tcp_stream,udp_stream,mqtt_topic,info,device_name,mapping_source,mapping_confidence,pcap_name,src_port,dst_port
0,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,-1.0,-1.0,1516,0x0010,...,1,-1.0,/iot/locator,"Publish Message (id=12526) [/iot/locator], Pub...",IoT_Thermostat,topic,high,normal_10.pcap,52976,1883
1,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,-1.0,-1.0,1516,0x0010,...,1,-1.0,/smarthome/thermostat,Publish Message (id=12535) [/smarthome/thermos...,IoT_Thermostat,topic,high,normal_10.pcap,52976,1883
2,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,-1.0,-1.0,523,0x0018,...,1,-1.0,/smarthome/garageDoor,Publish Message (id=12546) [/smarthome/garageD...,IoT_Thermostat,topic,high,normal_10.pcap,52976,1883
3,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,-1.0,-1.0,223,0x0018,...,1,-1.0,/smarthome/motionLights,Publish Message (id=12549) [/smarthome/motionL...,IoT_Motion_Light,topic,high,normal_10.pcap,52976,1883
4,1.554324e+09,192.168.1.152,3.122.49.24,6,52976,1883,-1.0,-1.0,350,0x0018,...,1,-1.0,/smarfactory/Modbus-data,Publish Message (id=12550) [/smarfactory/Modbu...,IoT_Motion_Light,topic,high,normal_10.pcap,52976,1883


In [27]:
pcap_series = work[pcap_col].astype(str)

default_flow_id = (
    pcap_series
    + "::P:" + work["ip_proto"].astype(str)
    + "::" + work["src_ip"]
    + "::" + work["src_port"].astype(str)
    + "::" + work["dst_ip"]
    + "::" + work["dst_port"].astype(str)
)

work["flow_id"] = default_flow_id

tcp_mask = (work["ip_proto"] == 6) & (work["tcp_stream"] != -1)
udp_mask = (work["ip_proto"] == 17) & (work["udp_stream"] != -1)

work.loc[tcp_mask, "flow_id"] = (
    pcap_series[tcp_mask]
    + "::TCPSTREAM::"
    + work.loc[tcp_mask, "tcp_stream"].astype(int).astype(str)
)

work.loc[udp_mask, "flow_id"] = (
    pcap_series[udp_mask]
    + "::UDPSTREAM::"
    + work.loc[udp_mask, "udp_stream"].astype(int).astype(str)
)

work["tcp_flags_int"] = work["tcp_flags"].apply(read_flag)

print("rows:", len(work))
print("unique flow_id:", work["flow_id"].nunique())

rows: 214040
unique flow_id: 14


In [28]:
flow_df = (
    work.groupby(["device_name", "flow_id"], as_index=False, dropna=False)
        .agg(
            first_time=("frame_time_epoch", "min"),
            last_time=("frame_time_epoch", "max"),
            IN_BYTES=("frame_len", "sum"),
            IN_PKTS=("frame_len", "size"),
            IPV4_DST_ADDR=("dst_ip", "first"),
            L4_DST_PORT=("dst_port", "first"),
            L4_SRC_PORT=("src_port", "first"),
            PROTOCOL=("ip_proto", "first"),
            SRC_TOS=("ip_dsfield", "first"),
            TCP_FLAGS=("tcp_flags_int", merge_flags),
            PCAP_GROUP=(pcap_col, "first"),
            MQTT_TOPIC=("mqtt_topic", "first"),
        )
)

flow_df["DURATION"] = flow_df["last_time"] - flow_df["first_time"]
flow_df["DURATION"] = pd.to_numeric(flow_df["DURATION"], errors="coerce").fillna(0.0)

flow_df["IPV4_DST_NET"] = flow_df["IPV4_DST_ADDR"].apply(to_ipv4_net)

print("flow_df shape:", flow_df.shape)
print(flow_df["device_name"].value_counts())
display(flow_df.head())

flow_df shape: (87, 16)
device_name
IoT_Modbus          14
IoT_Motion_Light    13
IoT_Fridge          13
IoT_Thermostat      13
IoT_Weather         13
IoT_Garage_Door     12
IoT_GPS_Tracker      9
Name: count, dtype: int64


,device_name,flow_id,first_time,last_time,IN_BYTES,IN_PKTS,IPV4_DST_ADDR,L4_DST_PORT,L4_SRC_PORT,PROTOCOL,SRC_TOS,TCP_FLAGS,PCAP_GROUP,MQTT_TOPIC,DURATION,IPV4_DST_NET
0,IoT_Fridge,normal_1.pcap::TCPSTREAM::3,1.554220e+09,1.554230e+09,789834,1928,3.122.49.24,1883,52976,6,0x02,24,normal_1.pcap,/smarthome/garageDoor,9458.245741,3.122.49.0/24
1,IoT_Fridge,normal_10.pcap::TCPSTREAM::1,1.554327e+09,1.554334e+09,1639,9,3.122.49.24,1883,52976,6,0x02,24,normal_10.pcap,/smarthome/garageDoor,7165.339273,3.122.49.0/24
2,IoT_Fridge,normal_11.pcap::TCPSTREAM::0,1.554343e+09,1.554343e+09,149,1,3.122.49.24,1883,52976,6,0x02,24,normal_11.pcap,/smarthome/fridge,0.000000,3.122.49.0/24
3,IoT_Fridge,normal_12.pcap::TCPSTREAM::0,1.554351e+09,1.554352e+09,1662,2,3.122.49.24,1883,52976,6,0x02,24,normal_12.pcap,/smarthome/fridge,590.085667,3.122.49.0/24
4,IoT_Fridge,normal_13.pcap::TCPSTREAM::1,1.554274e+09,1.554358e+09,34580,66,3.122.49.24,1883,52976,6,0x02,24,normal_13.pcap,/smarthome/fridge,84351.586651,3.122.49.0/24


In [29]:
flow_out = csv_path.with_name("meid20_flow_table_from_packets.csv")
flow_df.to_csv(flow_out, index=False, encoding="utf-8-sig")
print("saved:", flow_out)

saved: /content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/meid20_flow_table_from_packets.csv


In [30]:
work = flow_df.copy()

work["mid_time"] = (work["first_time"] + work["last_time"]) / 2.0
work = work.sort_values("mid_time").reset_index(drop=True)

idle_cut_idx = int(np.floor(0.2 * len(work)))
idle_cut_idx = max(idle_cut_idx, 1)

work["mode"] = "active"
work.loc[:idle_cut_idx - 1, "mode"] = "idle"

print(work["mode"].value_counts())
display(work[["device_name", "first_time", "last_time", "mid_time", "mode"]].head(10))
display(work[["device_name", "first_time", "last_time", "mid_time", "mode"]].tail(10))

mode
active    70
idle      17
Name: count, dtype: int64


,device_name,first_time,last_time,mid_time,mode
0,IoT_GPS_Tracker,1.554220e+09,1.554229e+09,1.554225e+09,idle
1,IoT_Fridge,1.554220e+09,1.554230e+09,1.554225e+09,idle
2,IoT_Garage_Door,1.554220e+09,1.554230e+09,1.554225e+09,idle
3,IoT_Motion_Light,1.554220e+09,1.554230e+09,1.554225e+09,idle
4,IoT_Thermostat,1.554220e+09,1.554230e+09,1.554225e+09,idle
5,IoT_Weather,1.554220e+09,1.554230e+09,1.554225e+09,idle
6,IoT_Modbus,1.554220e+09,1.554230e+09,1.554225e+09,idle
7,IoT_Fridge,1.554230e+09,1.554240e+09,1.554235e+09,idle
8,IoT_Modbus,1.554230e+09,1.554241e+09,1.554236e+09,idle
9,IoT_Motion_Light,1.554230e+09,1.554241e+09,1.554236e+09,idle


,device_name,first_time,last_time,mid_time,mode
77,IoT_Motion_Light,1.554335e+09,1.554347e+09,1.554341e+09,active
78,IoT_Modbus,1.554335e+09,1.554347e+09,1.554341e+09,active
79,IoT_Weather,1.554336e+09,1.554347e+09,1.554341e+09,active
80,IoT_Fridge,1.554343e+09,1.554343e+09,1.554343e+09,active
81,IoT_Fridge,1.554351e+09,1.554352e+09,1.554351e+09,active
82,IoT_Garage_Door,1.554352e+09,1.554352e+09,1.554352e+09,active
83,IoT_Thermostat,1.554347e+09,1.554358e+09,1.554353e+09,active
84,IoT_Motion_Light,1.554347e+09,1.554358e+09,1.554353e+09,active
85,IoT_Modbus,1.554347e+09,1.554358e+09,1.554353e+09,active
86,IoT_Weather,1.554347e+09,1.554358e+09,1.554353e+09,active


In [31]:
idle_df = work[work["mode"] == "idle"].copy()
active_df = work[work["mode"] == "active"].copy()

if len(idle_df) < 2 or len(active_df) < 2:
    raise ValueError("There are too few idle or active samples to continue partitioning")

idle_train, idle_test = split_in_time(idle_df, ratio=0.7)
active_train, active_test = split_in_time(active_df, ratio=0.7)

train_mix = pd.concat([idle_train, active_train], axis=0).reset_index(drop=True)
test_idle = idle_test.reset_index(drop=True)
test_active = active_test.reset_index(drop=True)
test_mix = pd.concat([idle_test, active_test], axis=0).reset_index(drop=True)

print("train_mix shape:", train_mix.shape)
print("test_idle shape:", test_idle.shape)
print("test_active shape:", test_active.shape)
print("test_mix shape:", test_mix.shape)

train_mix shape: (60, 18)
test_idle shape: (6, 18)
test_active shape: (21, 18)
test_mix shape: (27, 18)


In [32]:
print("train_mix class counts:")
print(train_mix["device_name"].value_counts())

print("\ntest_idle class counts:")
print(test_idle["device_name"].value_counts())

print("\ntest_active class counts:")
print(test_active["device_name"].value_counts())

print("\ntest_mix class counts:")
print(test_mix["device_name"].value_counts())

train_mix class counts:
device_name
IoT_Fridge          10
IoT_Modbus          10
IoT_Motion_Light     9
IoT_Weather          9
IoT_Thermostat       8
IoT_Garage_Door      7
IoT_GPS_Tracker      7
Name: count, dtype: int64

test_idle class counts:
device_name
IoT_Garage_Door     2
IoT_Thermostat      1
IoT_GPS_Tracker     1
IoT_Weather         1
IoT_Motion_Light    1
Name: count, dtype: int64

test_active class counts:
device_name
IoT_Thermostat      4
IoT_Modbus          4
IoT_Garage_Door     3
IoT_Motion_Light    3
IoT_Weather         3
IoT_Fridge          3
IoT_GPS_Tracker     1
Name: count, dtype: int64

test_mix class counts:
device_name
IoT_Thermostat      5
IoT_Garage_Door     5
IoT_Weather         4
IoT_Modbus          4
IoT_Motion_Light    4
IoT_Fridge          3
IoT_GPS_Tracker     2
Name: count, dtype: int64


In [33]:
use_mqtt_topic = False

feature_cols = [
    "DURATION",
    "IN_BYTES",
    "IN_PKTS",
    "IPV4_DST_NET",
    "L4_DST_PORT",
    "L4_SRC_PORT",
    "PROTOCOL",
    "SRC_TOS",
    "TCP_FLAGS",
]

cat_cols = [
    "IPV4_DST_NET",
    "L4_DST_PORT",
    "L4_SRC_PORT",
    "PROTOCOL",
    "SRC_TOS",
    "TCP_FLAGS",
]

num_cols = [
    "DURATION",
    "IN_BYTES",
    "IN_PKTS",
]

if use_mqtt_topic:
    feature_cols.append("MQTT_TOPIC")
    cat_cols.append("MQTT_TOPIC")

print("feature cols:", feature_cols)

feature cols: ['DURATION', 'IN_BYTES', 'IN_PKTS', 'IPV4_DST_NET', 'L4_DST_PORT', 'L4_SRC_PORT', 'PROTOCOL', 'SRC_TOS', 'TCP_FLAGS']


In [34]:
def clean_part(df):
    data = df.copy()

    data["device_name"] = data["device_name"].fillna("UNKNOWN_DEVICE").astype(str).str.strip()
    data.loc[data["device_name"] == "", "device_name"] = "UNKNOWN_DEVICE"

    for col in num_cols:
        data[col] = pd.to_numeric(data[col], errors="coerce").fillna(0.0)

    for col in cat_cols:
        data[col] = data[col].fillna("UNK").astype(str).str.strip()
        data.loc[data[col] == "", col] = "UNK"

    return data


train_mix_ready = clean_part(train_mix)
test_idle_ready = clean_part(test_idle)
test_active_ready = clean_part(test_active)
test_mix_ready = clean_part(test_mix)

display(train_mix_ready.head())

,device_name,flow_id,first_time,last_time,IN_BYTES,IN_PKTS,IPV4_DST_ADDR,L4_DST_PORT,L4_SRC_PORT,PROTOCOL,SRC_TOS,TCP_FLAGS,PCAP_GROUP,MQTT_TOPIC,DURATION,IPV4_DST_NET,mid_time,mode
0,IoT_GPS_Tracker,normal_1.pcap::TCPSTREAM::3,1.554220e+09,1.554229e+09,456275,1773,3.122.49.24,1883,52976,6,0x02,24,normal_1.pcap,/iot/locator,8334.428682,3.122.49.0/24,1.554225e+09,idle
1,IoT_Fridge,normal_1.pcap::TCPSTREAM::3,1.554220e+09,1.554230e+09,789834,1928,3.122.49.24,1883,52976,6,0x02,24,normal_1.pcap,/smarthome/garageDoor,9458.245741,3.122.49.0/24,1.554225e+09,idle
2,IoT_Garage_Door,normal_1.pcap::TCPSTREAM::3,1.554220e+09,1.554230e+09,323277,1151,3.122.49.24,1883,52976,6,0x02,24,normal_1.pcap,/smarthome/garageDoor,9742.448307,3.122.49.0/24,1.554225e+09,idle
3,IoT_Motion_Light,normal_1.pcap::TCPSTREAM::3,1.554220e+09,1.554230e+09,2394017,5713,3.122.49.24,1883,52976,6,0x02,152,normal_1.pcap,/smarthome/motionLights,9797.988803,3.122.49.0/24,1.554225e+09,idle
4,IoT_Thermostat,normal_1.pcap::TCPSTREAM::3,1.554220e+09,1.554230e+09,4103351,4686,3.122.49.24,1883,52976,6,0x02,152,normal_1.pcap,/smarthome/thermostat,9800.694951,3.122.49.0/24,1.554225e+09,idle


In [35]:
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", encoder, cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop"
)

lgbm_args = dict(
    objective="binary",
    class_weight="balanced",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)

In [36]:
def run_ovr_test(train_df, test_df):
    x_train = train_df[feature_cols].copy()
    y_train = train_df["device_name"].copy()

    x_test = test_df[feature_cols].copy()
    y_test = test_df["device_name"].copy()

    labels = sorted(set(y_train.unique()) & set(y_test.unique()))
    rows = []

    for name in labels:
        y_train_now = (y_train == name).astype(int)
        y_test_now = (y_test == name).astype(int)

        if y_train_now.sum() == 0 or y_test_now.sum() == 0:
            continue

        model = Pipeline([
            ("prep", preprocess),
            ("clf", LGBMClassifier(**lgbm_args)),
        ])

        model.fit(x_train, y_train_now)
        score = model.predict_proba(x_test)[:, 1]

        ap = average_precision_score(y_test_now, score)
        precision, recall, _ = precision_recall_curve(y_test_now, score)
        aucpr = auc(recall, precision)

        rows.append({
            "device_name": name,
            "AP": ap,
            "AUCPR": aucpr,
            "train_pos": int(y_train_now.sum()),
            "test_pos": int(y_test_now.sum()),
        })

    result = pd.DataFrame(rows)

    if result.empty:
        return result, {
            "macro_ap": np.nan,
            "weighted_ap": np.nan,
            "macro_aucpr": np.nan,
            "weighted_aucpr": np.nan,
        }

    result = result.sort_values("AUCPR", ascending=False).reset_index(drop=True)

    summary = {
        "macro_ap": result["AP"].mean(),
        "weighted_ap": np.average(result["AP"], weights=result["test_pos"]),
        "macro_aucpr": result["AUCPR"].mean(),
        "weighted_aucpr": np.average(result["AUCPR"], weights=result["test_pos"]),
    }

    return result, summary

In [37]:
idle_res, idle_sum = run_ovr_test(train_mix_ready, test_idle_ready)
active_res, active_sum = run_ovr_test(train_mix_ready, test_active_ready)
mix_res, mix_sum = run_ovr_test(train_mix_ready, test_mix_ready)

[LightGBM] [Info] Number of positive: 7, number of negative: 53
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000169 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 62
[LightGBM] [Info] Number of data points in the train set: 60, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 9, number of negative: 51
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000023 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 62
[LightGBM] [Info] Number of data points in the train set: 60, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 9, number of negative: 51
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002589 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 62
[LightGBM] [Info] Number of data points in the train set: 60, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 7, number of negative: 53
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000025 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 62
[LightGBM] [Info] Number of data points in the train set: 60, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [38]:
summary_table = pd.DataFrame([
    {
        "Scenario": "Train: Mix -> Test: Idle",
        "Macro_AP": idle_sum["macro_ap"],
        "Macro_AUCPR": idle_sum["macro_aucpr"],
        "Weighted_AP": idle_sum["weighted_ap"],
        "Weighted_AUCPR": idle_sum["weighted_aucpr"],
    },
    {
        "Scenario": "Train: Mix -> Test: Active",
        "Macro_AP": active_sum["macro_ap"],
        "Macro_AUCPR": active_sum["macro_aucpr"],
        "Weighted_AP": active_sum["weighted_ap"],
        "Weighted_AUCPR": active_sum["weighted_aucpr"],
    },
    {
        "Scenario": "Train: Mix -> Test: Mix",
        "Macro_AP": mix_sum["macro_ap"],
        "Macro_AUCPR": mix_sum["macro_aucpr"],
        "Weighted_AP": mix_sum["weighted_ap"],
        "Weighted_AUCPR": mix_sum["weighted_aucpr"],
    },
])

display(summary_table)

,Scenario,Macro_AP,Macro_AUCPR,Weighted_AP,Weighted_AUCPR
0,Train: Mix -> Test: Idle,0.966667,0.958333,0.944444,0.930556
1,Train: Mix -> Test: Active,0.577948,0.647120,0.639739,0.673437
2,Train: Mix -> Test: Mix,0.580822,0.633540,0.614176,0.649570


In [39]:
out_dir = flow_out.parent

summary_file = out_dir / "temporal20_idle80_active_summary.csv"
idle_file = out_dir / "temporal20_idle80_active_idle_per_device.csv"
active_file = out_dir / "temporal20_idle80_active_active_per_device.csv"
mix_file = out_dir / "temporal20_idle80_active_mix_per_device.csv"

summary_table.to_csv(summary_file, index=False, encoding="utf-8-sig")
idle_res.to_csv(idle_file, index=False, encoding="utf-8-sig")
active_res.to_csv(active_file, index=False, encoding="utf-8-sig")
mix_res.to_csv(mix_file, index=False, encoding="utf-8-sig")

print("saved:")
print(summary_file)
print(idle_file)
print(active_file)
print(mix_file)

saved:
/content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/temporal20_idle80_active_summary.csv
/content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/temporal20_idle80_active_idle_per_device.csv
/content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/temporal20_idle80_active_active_per_device.csv
/content/drive/MyDrive/iot/iot device name/network/packet_csvs_device_only/temporal20_idle80_active_mix_per_device.csv
